# Emotion-Aware Chatbot Training with BERT on Colab

In [ ]:
!pip install transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:
!pip install wandb

In [ ]:
# Import libraries
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, DefaultFlowCallback
import torch
from datasets import Dataset
import os
from google.colab import files


# Upload files
uploaded = files.upload()

# Load and combine CSVs
df1 = pd.read_csv("goemotions_1.csv")
df2 = pd.read_csv("goemotions_2.csv")
df3 = pd.read_csv("goemotions_3.csv")
df = pd.concat([df1, df2, df3], ignore_index=True)

# Extract label
emotion_columns = df.columns[9:]
def extract_label(row):
    for col in emotion_columns:
        if row[col] == 1:
            return col
    return "neutral"

df["emotion"] = df.apply(extract_label, axis=1)
df = df[["text", "emotion"]]

# Label encode
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["emotion"])

# Convert to HF dataset
dataset = Dataset.from_pandas(df)

# Tokenization
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Split for eval
split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

# Load model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=len(encoder.classes_))
model.to("cuda")

# Training arguments
training_args = TrainingArguments(
    output_dir="./emotion_model_csv",
    evaluation_strategy="epoch",
    per_device_train_batch_size=32,
    num_train_epochs=2,
    save_strategy="epoch",
    logging_dir="./logs_csv",
    fp16=True,
    report_to="none" # disable all integrations
)


# Disable WandB
os.environ["WANDB_DISABLED"] = "true"

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer
)

# Train
trainer.train()

# Save and download
model.save_pretrained("emotion_model_csv")
tokenizer.save_pretrained("emotion_model_csv")
!zip -r emotion_model_csv.zip emotion_model_csv
files.download("emotion_model_csv.zip")


Saving goemotions_1.csv to goemotions_1 (1).csv
Saving goemotions_2.csv to goemotions_2 (1).csv
Saving goemotions_3.csv to goemotions_3 (1).csv


<ipython-input-5-54f18dec9a14>:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["label"] = encoder.fit_transform(df["emotion"])


Map:   0%|          | 0/211225 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-5-54f18dec9a14>:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,1.861100,1.832252
2,1.724500,1.808403


  adding: emotion_model_csv/ (stored 0%)
  adding: emotion_model_csv/tokenizer_config.json (deflated 75%)
  adding: emotion_model_csv/model.safetensors (deflated 7%)
  adding: emotion_model_csv/vocab.txt (deflated 53%)
  adding: emotion_model_csv/checkpoint-11882/ (stored 0%)
  adding: emotion_model_csv/checkpoint-11882/optimizer.pt (deflated 17%)
  adding: emotion_model_csv/checkpoint-11882/tokenizer_config.json (deflated 75%)
  adding: emotion_model_csv/checkpoint-11882/model.safetensors (deflated 7%)
  adding: emotion_model_csv/checkpoint-11882/vocab.txt (deflated 53%)
  adding: emotion_model_csv/checkpoint-11882/training_args.bin (deflated 52%)
  adding: emotion_model_csv/checkpoint-11882/rng_state.pth (deflated 25%)
  adding: emotion_model_csv/checkpoint-11882/scheduler.pt (deflated 55%)
  adding: emotion_model_csv/checkpoint-11882/special_tokens_map.json (deflated 42%)
  adding: emotion_model_csv/checkpoint-11882/scaler.pt (deflated 60%)
  adding: emotion_model_csv/checkpoint-118

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>